In [5]:
using Serialization, Distributions, StableRNGs, LogExpFunctions, Statistics

Calculate Quantiles of Experimental Quantities - Coursegrain save at

In [6]:
data_files = readdir("data")
sort_files = data_files .!= ".DS_Store"
data_files = sort(data_files[sort_files]) #sort to ensure consistent order
sort_files = data_files .!= "predicted_kd.csv"
data_files = sort(data_files[sort_files])
quantity_names = [replace(data_files[i], ".csv" => "") for i in 1:length(data_files)];

In [3]:
augmented_run = "204"
baseline_run = "205"
for r in [augmented_run, baseline_run]    
    for i in quantity_names
        predictions = Base.stack(deserialize("outputs/500_$(r)_predictions_$(i).jls")) #ntime x n
        n_time_points, n_predictions = size(predictions)
        uq = 0.95
        lq = 0.05
        m = 0.5
        quantile_up = [Statistics.quantile(predictions[i,:], uq) for i in 1:n_time_points]
        quantile_low = [Statistics.quantile(predictions[i,:], lq) for i in 1:n_time_points]
        median = [Statistics.quantile(predictions[i,:], m) for i in 1:n_time_points]
        serialize("outputs/600_$(r)_prediction_quantiles_coursegrain_$(i).jls", Dict("lower_quantile"=>quantile_low, 
        "median"=>median, "upper_quantile" =>quantile_up))
    end
end

In [7]:
augmented_run = "204"
baseline_run = "205"
for r in [augmented_run, baseline_run]
    SSE_string = ["\nSSE\n"]    
    for i in quantity_names
        if i == "active_g_dose_response" || i == "rl_dose_response"
            median_predictions = deserialize("outputs/600_$(r)_prediction_quantiles_coursegrain_$(i).jls")["median"][1:end-1] #exclude dose 1000, not in experimental data
            data = deserialize("outputs/000_processed_$(i).dict")["response"]
            SSE = string(sum((median_predictions .- data).^2))
            text_out = "$(i):" * "$(SSE)\n"
            push!(SSE_string, text_out)
        else 
            median_predictions = deserialize("outputs/600_$(r)_prediction_quantiles_coursegrain_$(i).jls")["median"]
            data = deserialize("outputs/000_processed_$(i).dict")["response"]
            SSE = string(sum((median_predictions .- data).^2))
            text_out = "$(i):" * "$(SSE)\n"
            push!(SSE_string,text_out)
        end
    end
    serialize("outputs/600_$(r)_SSE.txt", join(SSE_string))
end

In [11]:
augmented_run = "200"
baseline_run = "201"
for r in [augmented_run, baseline_run]
    SSE_string = ["\nRMSE\n"]    
    for i in quantity_names
        if i == "active_g_dose_response" || i == "rl_dose_response"
            median_predictions = deserialize("outputs/600_$(r)_prediction_quantiles_coursegrain_$(i).jls")["median"][1:end-1] #exclude dose 1000, not in experimental data
            data = deserialize("outputs/000_processed_$(i).dict")["response"]
            n_points = length(data)
            SSE = string(sqrt(sum((median_predictions .- data).^2)/n_points))
            text_out = "$(i):" * "$(SSE)\n"
            push!(SSE_string, text_out)
        else 
            median_predictions = deserialize("outputs/600_$(r)_prediction_quantiles_coursegrain_$(i).jls")["median"]
            data = deserialize("outputs/000_processed_$(i).dict")["response"]
            n_points = length(data)
            SSE = string(sqrt(sum((median_predictions .- data).^2)/n_points))
            text_out = "$(i):" * "$(SSE)\n"
            push!(SSE_string,text_out)
        end
    end
    serialize("outputs/600_$(r)_RMSE.txt", join(SSE_string))
end